# TxGNN on two relations only

Everything about TxGNN is untouched except **which relations exist in the graph**. The arm of interest
keeps only the three drug-disease task relations and the two relations the disease signature reads:

```
disease | disease_disease       | disease           <- read by metric learning
disease | rev_disease_protein   | gene/protein      <- read by metric learning
gene/protein | disease_protein  | disease
drug | indication               | disease           <- task label
drug | contraindication         | disease           <- task label
drug | off-label use            | disease           <- task label
disease | rev_indication        | drug
disease | rev_contraindication  | drug
disease | rev_off-label use     | drug
```

**9 edge types of 50. 41 dropped.** About **3.9%** of the training edges survive (~260k of ~6.73M):
`disease_disease` 53,522 + `disease_protein`/`rev_` 66,841 each + 72,648 drug-disease task edges.
`drug_drug` (2.2M), `anatomy_protein_present` (1.26M each way), `protein_protein` (534k) and 35 other
edge types are gone.

### Metric learning needs no change, and that is the point

`DistMultPredictor` already builds its signature from exactly `['disease_disease',
'rev_disease_protein']` (`model.py:88`). Those edges are **kept in full here**, so the signature is
bit-identical to the full-KG baseline's - `run_one` asserts its dimension and maximum match. No
monkeypatch, no `disease_etypes` edit. This notebook therefore isolates one question:

> Does the other 96% of the knowledge graph contribute anything to message passing, beyond the two
> relations metric learning already reads?

### Why this is worth running

`ablation_stage_a` removed each relation **one at a time** and found 20 of 22 individually worthless
(<= 0.014 AUPRC, none significant). That does not rule out the 41 dropped edge types mattering
*collectively* - leave-one-out is blind to redundancy. This arm removes them all at once.

### Arms

| arm | edge types | train edges | note |
|---|---|---|---|
| `full_kg` | 50 | ~6.73M | the reference; imported from existing grids, trains nothing |
| `only_disease_relations` | 9 | ~260k (3.9%) | **the arm of interest** |
| `dd_only` | 6 | ~73k (1.1%) | control: task labels alone, no KG context at all |

`dd_only` is the floor. It matters because "two relations match the full KG" is only interesting
alongside "and those two relations are doing real work". Note its signature is necessarily **all zero**
(neither source relation is in the graph), so it is a degenerate metric-learning condition - the
notebook asserts that and section 8 says how to read it. Set `INCLUDE_DD_ONLY_CONTROL = False` to skip.

### Results and resumability

`results/ablation_keep_disease_only/<run_key>/`, written the moment each run finishes; runs with both
output files are skipped. Safe to re-run from cell 1 after a crash, kernel restart or interrupt.

---
## 1 - Environment

In [ ]:
import os, sys, json, time, gc, random, shutil, inspect, platform, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')


def _find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'pyg_implementation' / 'txgnn').is_dir():
            return cand
    raise RuntimeError('Could not find pyg_implementation/txgnn above ' + str(p))


REPO_ROOT = _find_repo_root()
PYG_ROOT  = REPO_ROOT / 'pyg_implementation'

# Put the PyG port FIRST on sys.path so `import txgnn` never resolves to dgl_implementation/.
sys.path.insert(0, str(PYG_ROOT))

import torch
import txgnn
from txgnn import TxData, TxGNN
from txgnn.utils import evaluate_fb, create_pyg_graph, get_n_params

assert 'pyg_implementation' in txgnn.__file__, (
    'Wrong txgnn package on sys.path (%s). Restart the kernel.' % txgnn.__file__)

# Guard for requirement 1: the Gumbel-gate switch was removed from the library in commit 2d064cc.
# Passing it (or reading m.config['use_decision_network'], or asserting on m.model.pred.block_mode)
# the way the old notebook's cell 43 did is a TypeError / KeyError now. Fail here, loudly, rather
# than eight hours into the grid.
_MI_PARAMS = set(inspect.signature(TxGNN.model_initialize).parameters)
assert 'use_decision_network' not in _MI_PARAMS, (
    'model_initialize accepts use_decision_network again - revisit MODEL_CFG before trusting this grid.')


def set_all_seeds(seed: int):
    '''Seed every RNG the training path touches.'''
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

print('repo root     :', REPO_ROOT)
print('txgnn package :', txgnn.__file__)
print('torch         :', torch.__version__)
print('device        :', DEVICE, '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else platform.processor())
print('model_initialize params:', sorted(_MI_PARAMS - {'self'}))
if DEVICE == 'cpu':
    print('\nWARNING: CPU only. A ~45 GPU-hour grid is not practical here.')

---
## 2 - Configuration

Model and schedule are identical to every other ablation grid in this repo, so the deltas are directly
comparable: `n_hid=n_inp=n_out=100`, `proto=True`, `proto_num=3`, `sim_measure='all_nodes_profile'`,
`agg_measure='rarity'`, `attention=False`; pretrain 1 epoch at lr 1e-3; finetune 1000 epochs at lr 5e-4
with `valid_per_n=20`. `train_print_per_n` controls console logging only.

`SEEDS` is **exactly** the 20-seed set of `ablation_encoder_swap`, which means every `full_kg` baseline
imports from `ablation_encoder_swap/rgcn_seed<n>` and nothing has to be retrained for the reference arm.

These arms are far cheaper than the earlier grids. Pretraining is one pass over the *remaining* training
edges in batches of 1,024, so dropping 96% of the edges cuts it from ~6,578 steps to ~250 - minutes
instead of ~19 min per run. The plan cell projects the real figure from the measured baseline timings.

In [ ]:
SPLIT   = 'complex_disease'
SEEDS   = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 31, 41, 42, 63, 70, 77, 86, 89, 95, 98]
# ^ exactly the seed set of ablation_encoder_swap, so all 20 full-KG baselines import free
PROFILE = 'full'

# verbatim from the 'full' profile / reproduce/train.py - no use_decision_network (see cell above)
MODEL_CFG = dict(
    n_hid       = 100,
    n_inp       = 100,
    n_out       = 100,
    proto       = True,                  # disease-similarity metric learning
    proto_num   = 3,
    attention   = False,
    sim_measure = 'all_nodes_profile',   # signature = binary vector over disease + gene/protein neighbours
    agg_measure = 'rarity',
)

PRETRAIN_CFG = dict(n_epoch=1,    learning_rate=1e-3, batch_size=1024, train_print_per_n=500)
FINETUNE_CFG = dict(n_epoch=1000, learning_rate=5e-4, train_print_per_n=100, valid_per_n=20)

DD_REL     = ['contraindication', 'indication', 'off-label use']
DD_REV_REL = ['rev_' + r for r in DD_REL]
DD_ETYPES  = [('drug', r, 'disease') for r in DD_REL] + [('disease', r, 'drug') for r in DD_REV_REL]

DATA_FOLDER = REPO_ROOT / 'data'
ABL_ROOT    = REPO_ROOT / 'results' / 'ablation_keep_disease_only'
ABL_ROOT.mkdir(parents=True, exist_ok=True)

# sibling grids, for the full-KG baseline import and the section 7.1 context table
STAGE_A_ROOT   = REPO_ROOT / 'results' / 'ablation_stage_a'
REPLICATION_ROOT = REPO_ROOT / 'results' / 'ablation_stage_a_seeds10_19'
SIGNATURE_ROOT = REPO_ROOT / 'results' / 'ablation_signature_only'
ENCODER_ROOT   = REPO_ROOT / 'results' / 'ablation_encoder_swap'

# ablation_encoder_swap/rgcn covers every seed in SEEDS, so it is tried first
BASELINE_SOURCES = [
    (ENCODER_ROOT, 'rgcn'),
    (REPLICATION_ROOT, 'baseline'),
    (STAGE_A_ROOT, 'baseline'),
]

# --- switches ------------------------------------------------------------------------------------
RUN_GRID          = True   # section 6.2 trains; set False to inspect the plan / aggregate only
SAVE_MODELS       = False
IMPORT_BASELINE   = True   # reuse a sibling grid's full-KG baseline - see section 5
INCLUDE_DD_ONLY_CONTROL = True   # the 6-edge-type floor arm; see section 3
STOP_AFTER_N_RUNS = None   # e.g. 2 for a short unattended smoke pass over the harness

print('split   :', SPLIT)
print('seeds   :', SEEDS)
print('results :', ABL_ROOT)
print('pretrain:', PRETRAIN_CFG)
print('finetune:', FINETUNE_CFG)

---
## 3 - The arms

`keep` lists relation **base names**; the helper expands each to `{rel, 'rev_' + rel}`, which is correct
for both hetero relations (`disease_protein` -> plus `rev_disease_protein`) and same-type ones
(`disease_disease` has no `rev_` twin because `utils.reverse_rel_generation` only renames when
`x_type != y_type`, so both directions already live under the one name).

`keep=None` means "keep everything" and is the `full_kg` reference.

`expect_etypes` is asserted against `len(G.edge_types)` after filtering, so a typo in a relation name
fails immediately instead of quietly producing a differently-sized graph.

In [ ]:
# the supervised relations - these can never be dropped, they are the task
DD_RELATIONS = ['indication', 'contraindication', 'off-label use']

# the two relations DistMultPredictor's signature reads (model.py:88)
SIGNATURE_RELATIONS = ['disease_disease', 'disease_protein']


def _make_arm(name, keep, expect_etypes, note=''):
    return {'arm': name, 'keep': keep, 'expect_etypes': expect_etypes, 'note': note}


ARMS = [
    _make_arm('full_kg', None, 50, 'reference: the unmodified KG'),
    _make_arm('only_disease_relations', DD_RELATIONS + SIGNATURE_RELATIONS, 9,
              'task labels + the two relations metric learning reads'),
]
if INCLUDE_DD_ONLY_CONTROL:
    # floor: no KG context at all. The signature has no source relation left, so it is all-zero -
    # a degenerate metric-learning condition, asserted in run_one and discussed in section 8.
    ARMS.append(_make_arm('dd_only', list(DD_RELATIONS), 6,
                          'control/floor: task labels only, signature necessarily empty'))

ARMS_BY_NAME = {a['arm']: a for a in ARMS}
assert len(ARMS_BY_NAME) == len(ARMS), 'duplicate arm name'
assert 'full_kg' in ARMS_BY_NAME, 'the reference arm is required for paired deltas'

print('%d arms x %d seeds = %d runs\n' % (len(ARMS), len(SEEDS), len(ARMS) * len(SEEDS)))
for a in ARMS:
    print('  %-24s edge types %-3d  keep=%s'
          % (a['arm'], a['expect_etypes'], 'ALL' if a['keep'] is None else a['keep']))
    print('  %-24s %s' % ('', a['note']))

---
## 4 - Mechanics

**`keep_relations(df, keep)`** filters a split frame to the surviving relation names. **`KeptData`** is a
`TxData` stand-in holding the three filtered frames and a graph rebuilt with
`create_pyg_graph(df_train, base['df'])`. The second argument is the **untouched** full KG frame, and
`create_pyg_graph` derives `num_nodes` per type from it, so **every node type and every node still
exists** with identical indices - `anatomy`, `pathway`, `exposure` and the rest simply become isolated.
`HeteroRGCNLayer` fills zeros for node types with no incoming edges (`model.py:424-429`), so that is a
supported state, and it keeps node counts and the signature dimension comparable with the full-KG
baseline.

**No signature patch.** Unlike `ablation_signature_only.ipynb`, nothing here touches
`obtain_disease_profile` or `disease_etypes`. The two signature relations are kept in full, so the
profiles are bit-identical to the baseline's. `run_one` verifies this rather than assuming it: it probes
the built profiles and asserts the dimension equals `G['disease'].num_nodes + G['gene/protein'].num_nodes`
and the maximum is `1.0`. For `dd_only` it asserts the maximum is `0.0` instead, since neither source
relation survives.

### Two confounds to keep in mind when reading the result

1. **Encoder capacity shrinks with the relation count.** `HeteroRGCNLayer` holds one
   `nn.Linear(100, 100)` per relation, so the encoder goes from 2 x 50 x 10,100 = **1,010,000**
   parameters at 50 edge types to 2 x 9 x 10,100 = **181,800** at 9, and **121,200** at 6. The
   encoder-swap grid showed capacity matters a lot on this metric (collapsing 50 relation matrices to
   one shared matrix cost -0.2266 indication), so a loss here is **not** attributable to missing KG
   information alone. `run_one` records `encoder_params` so the two effects can at least be reported
   side by side.
2. **"These two relations" vs "any nine edge types".** A matched control that keeps the task labels plus
   three *arbitrary* non-disease relations of similar size would separate the identity of the relations
   from the mere count. Not built here; specified in section 8.

In [ ]:
SIG_ETYPES = ['disease_disease', 'rev_disease_protein']   # the library default, model.py:88
SIG_NODES  = ['disease', 'gene/protein']


def expand_relations(names):
    '''Relation base names -> the names actually present in the frames, incl. rev_ twins.'''
    out = set()
    for r in names:
        out |= {r, 'rev_' + r}
    return out


def keep_relations(df, keep):
    '''Filter a split frame down to `keep` (plus rev_ twins). `None` keeps everything.'''
    if keep is None:
        return df.reset_index(drop=True)
    return df[df.relation.isin(expand_relations(keep))].reset_index(drop=True)


class KeptData:
    '''Minimal TxData stand-in holding the reduced splits and a rebuilt graph.'''

    def __init__(self, base, keep, data_folder, split):
        # base['df'] is the untouched full KG: create_pyg_graph reads num_nodes from it, so every
        # node type survives with identical indices and only edges disappear
        self.df = base['df']
        self.df_train = keep_relations(base['df_train'], keep)
        self.df_valid = keep_relations(base['df_valid'], keep)
        self.df_test  = keep_relations(base['df_test'],  keep)
        self.G = create_pyg_graph(self.df_train, self.df)
        self.data_folder = str(data_folder)
        self.split = split
        self.disease_eval_idx = None
        self.no_kg = False


def encoder_params(model):
    return (sum(p.numel() for p in model.layer1.parameters())
            + sum(p.numel() for p in model.layer2.parameters()))


def signature_probe(model):
    '''(dim, max) of the disease signatures actually built. max == 0 => every signature is zero.'''
    try:
        prof = model.pred.diseases_profile_etypes
        et = next(iter(prof))
        vals = list(prof[et].values())[:64]
        if not vals:
            return None, None
        return int(vals[0].shape[0]), float(max(float(v.max()) for v in vals))
    except Exception:
        return None, None


# --- per-seed split cache -------------------------------------------------------------------------
_SEED_CACHE = {}


def get_seed_data(seed):
    '''Load (and cache) the unablated split frames for one seed.

    Holds one seed at a time - `df` is millions of rows. prepare_split() caches to disk, so the
    first call per seed writes data/<split>_<seed>/ and later calls just read the CSVs.
    '''
    if seed in _SEED_CACHE:
        return _SEED_CACHE[seed]
    _SEED_CACHE.clear()
    gc.collect()
    print('[data] loading split seed %d ...' % seed)
    t0 = time.time()
    set_all_seeds(seed)
    d = TxData(data_folder_path=str(DATA_FOLDER))
    d.prepare_split(split=SPLIT, seed=seed, no_kg=False)
    base = {'df': d.df, 'df_train': d.df_train, 'df_valid': d.df_valid, 'df_test': d.df_test}
    del d
    gc.collect()
    _SEED_CACHE[seed] = base
    print('[data] seed %d ready in %.1f s | train edges %d'
          % (seed, time.time() - t0, len(base['df_train'])))
    return base


def run_dir(arm, seed):
    return ABL_ROOT / ('%s_seed%d' % (arm, seed))


# --- reusing a sibling grid's baseline ------------------------------------------------------------
MODEL_KEYS_CHECKED = sorted(MODEL_CFG)       # the keys this notebook actually sets


def import_matching_run(src, dst, expect, extra=None):
    '''Copy a finished run from another grid iff its metadata matches `expect`. Returns (ok, why).'''
    sm, se = src / 'run_meta.json', src / 'edge_level_test_metrics.csv'
    if not (sm.exists() and se.exists()):
        return False, 'no run at %s' % src.name
    meta = json.loads(sm.read_text())
    for k, want in expect.items():
        got = meta.get(k, '<missing>')
        if isinstance(want, dict):
            if not isinstance(got, dict):
                return False, '%s is not a dict' % k
            bad = {kk: (got.get(kk, '<missing>'), vv) for kk, vv in want.items()
                   if got.get(kk, '<missing>') != vv}
            if bad:
                return False, '%s mismatch %s' % (k, bad)
        elif got != want:
            return False, '%s mismatch: %r != %r' % (k, got, want)

    dst.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(se, dst / 'edge_level_test_metrics.csv')
    meta.update(extra or {})
    meta['imported_from'] = str(src.relative_to(REPO_ROOT))
    meta['run_key'] = dst.name
    meta['arm'] = dst.name.rsplit('_seed', 1)[0]
    with open(dst / 'run_meta.json', 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return True, 'imported from %s' % src.relative_to(REPO_ROOT)


def try_import_baseline(seed):
    '''Look through BASELINE_SOURCES for a full-KG run on this seed with an identical config.'''
    if not IMPORT_BASELINE:
        return False, 'disabled'
    expect = {'split': SPLIT, 'seed': seed,
              'dropped_edges_total': 0,
              'signature_disease_etypes': ['disease_disease', 'rev_disease_protein'],
              'model_config': {k: MODEL_CFG[k] for k in MODEL_KEYS_CHECKED},
              'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG}
    whys = []
    for root, arm in BASELINE_SOURCES:
        ok, why = import_matching_run(
            root / ('%s_seed%d' % (arm, seed)), run_dir('full_kg', seed), expect,
            extra={'experiment': 'keep_disease_only', 'kept_relations_requested': None,
                   'dropped_edges_total': 0, 'signature_sources_present': SIG_ETYPES,
                   'train_edges_kept_pct': 100.0, 'proto': True})
        if ok:
            return True, why
        whys.append('%s/%s_seed%d: %s' % (root.name, arm, seed, why))
    return False, ' | '.join(whys)


def is_done(arm, seed):
    d = run_dir(arm, seed)
    return (d / 'edge_level_test_metrics.csv').exists() and (d / 'run_meta.json').exists()

---
## 5 - Plan

Prints the kept/dropped edge counts per arm from the real split, the expected edge-type count, the
encoder parameter count implied by that, and a time projection scaled by the surviving edge fraction
(pretraining is one pass over the remaining training edges, `TxGNN.py:141-148`).

**Baseline import.** Each missing `full_kg_seed<n>` is offered the matching run from
`ablation_encoder_swap/rgcn`, then the replication grid's `baseline`, then Stage A's. A run is accepted
only if its recorded `split`, `seed`, `model_config` (the eight keys this notebook sets), `pretrain`,
`finetune`, `dropped_edges_total == 0` and `signature_disease_etypes` all match, and the copy keeps an
`imported_from` field. Because `SEEDS` is the encoder-swap seed set, all 20 should import and the
reference arm costs nothing.

Splits for every seed here already exist on disk from the earlier grids, so `prepare_split` just reads
the cached CSVs.

In [ ]:
# --- offer each missing full-KG seed a sibling grid's run before planning anything ---------------
if IMPORT_BASELINE:
    for _s in SEEDS:
        if is_done('full_kg', _s):
            continue
        _ok, _why = try_import_baseline(_s)
        print('[import] full_kg_seed%d: %s' % (_s, _why if _ok else 'will train'))
        if not _ok:
            print('          %s' % _why)

BASE_PRETRAIN_S, BASE_FINETUNE_S = 1155.7, 316.5   # fallback: the seed-1 full run
_ref = REPO_ROOT / 'results' / ('TxGNN_%s_seed1_%s' % (SPLIT, PROFILE)) / 'run_config.json'
if _ref.exists():
    _r = json.loads(_ref.read_text())
    BASE_PRETRAIN_S = _r.get('pretrain_seconds', BASE_PRETRAIN_S)
    BASE_FINETUNE_S = _r.get('finetune_seconds', BASE_FINETUNE_S)
print('full-KG reference timing: pretrain %.1f min + finetune %.1f min = %.1f min/run\n'
      % (BASE_PRETRAIN_S / 60, BASE_FINETUNE_S / 60, (BASE_PRETRAIN_S + BASE_FINETUNE_S) / 60))

_base0 = get_seed_data(SEEDS[0])
_train = _base0['df_train']
_present = set(_train.relation.unique())
N_TRAIN_EDGES = len(_train)

# every relation named in section 3 must exist, or the arm would silently keep fewer edge types
_named = set(DD_RELATIONS) | set(SIGNATURE_RELATIONS)
_missing = sorted(r for r in _named if r not in _present and ('rev_' + r) not in _present)
assert not _missing, 'relations absent from df_train (seed %d): %s' % (SEEDS[0], _missing)

rows, total_s = [], 0.0
for a in ARMS:
    kept = len(keep_relations(_train, a['keep']))
    frac = kept / N_TRAIN_EDGES
    est_s = (BASE_PRETRAIN_S + BASE_FINETUNE_S) * frac
    done = [s for s in SEEDS if is_done(a['arm'], s)]
    todo = [s for s in SEEDS if s not in done]
    total_s += est_s * len(todo)
    rows.append({'arm': a['arm'],
                 'edge types (expected)': a['expect_etypes'],
                 'encoder params': 2 * a['expect_etypes'] * (100 * 100 + 100),
                 'train edges kept': kept,
                 'pct of KG': 100.0 * frac,
                 'est min/run': est_s / 60,
                 'done': len(done), 'todo': len(todo)})

plan = pd.DataFrame(rows)
print('train edges (seed %d, full KG): %d\n' % (SEEDS[0], N_TRAIN_EDGES))
display(plan.style.format({'pct of KG': '{:.2f}', 'est min/run': '{:.1f}',
                           'encoder params': '{:,.0f}', 'train edges kept': '{:,.0f}'})
                  .hide(axis='index'))

RUN_KEYS = [(a['arm'], s) for s in SEEDS for a in ARMS]     # seed-major, matches the run order
pending = [k for k in RUN_KEYS if not is_done(*k)]
print('\nrun_keys (%d total, %d done, %d pending):'
      % (len(RUN_KEYS), len(RUN_KEYS) - len(pending), len(pending)))
for arm, seed in RUN_KEYS:
    print('  [%s] %s_seed%d' % ('x' if is_done(arm, seed) else ' ', arm, seed))
print('\nestimated remaining: %.1f GPU-hours' % (total_s / 3600))
print('order: seed-major - after seed %d finishes every arm has n=1 and is already paired.' % SEEDS[0])

---
## 6 - The grid

Each finished run writes `edge_level_test_metrics.csv` and `run_meta.json` into
`results/ablation_keep_disease_only/<run_key>/`. The metadata records the relations kept, the full
`sorted(G.edge_types)` after filtering, kept/dropped edge counts, the probed signature dimension and
maximum, encoder and total parameter counts, and timings.

`MICRO (drug-disease)` and `MACRO (drug-disease)` are computed over the six drug-disease edge types only
and are comparable across arms. `MACRO (all relations)` is **not** - `get_all_metrics_fb(full_mode=True)`
averages over `G.edge_types`, which is 50, 9 or 6 depending on the arm. It is written to the CSV with a
label saying so, and the aggregation ignores it.

In [ ]:
def run_one(arm_name, seed, verbose=True):
    '''Train one (arm, seed) and write its results. Returns the edge-level DataFrame.'''
    arm  = ARMS_BY_NAME[arm_name]
    rdir = run_dir(arm_name, seed)
    el_csv, meta_json = rdir / 'edge_level_test_metrics.csv', rdir / 'run_meta.json'

    if el_csv.exists() and meta_json.exists():
        if verbose:
            print('[skip] %s_seed%d -> cached' % (arm_name, seed))
        return pd.read_csv(el_csv)

    if arm_name == 'full_kg':
        ok, why = try_import_baseline(seed)
        if ok:
            print('[skip] full_kg_seed%d -> %s' % (seed, why))
            return pd.read_csv(el_csv)

    print('\n' + '#' * 78)
    print('# %s   seed %d   keep=%s'
          % (arm_name, seed, 'ALL (full KG)' if arm['keep'] is None else arm['keep']))
    print('#' * 78)

    base     = get_seed_data(seed)
    n_before = len(base['df_train'])

    set_all_seeds(seed)
    d = KeptData(base, arm['keep'], DATA_FOLDER, SPLIT)
    etypes_after = sorted('%s|%s|%s' % et for et in d.G.edge_types)
    rel_names = {et[1] for et in d.G.edge_types}

    # --- the graph is what we think it is -------------------------------------------------------
    assert len(etypes_after) == arm['expect_etypes'], \
        'expected %d edge types, got %d: %s' % (arm['expect_etypes'], len(etypes_after), etypes_after)
    missing_dd = [r for r in DD_RELATIONS + ['rev_' + r for r in DD_RELATIONS] if r not in rel_names]
    assert not missing_dd, 'the supervised relations must survive, missing: %s' % missing_dd
    # Node types and counts come from the untouched full df, not from the surviving edges, so a
    # reduced graph must still carry every node type the full KG has. create_pyg_graph derives them
    # from df.x_type / df.y_type and always forces 'effect/phenotype' in (utils.py:1102-1112).
    n_node_types = len(d.G.node_types)
    expect_ntypes = set(base['df'].x_type) | set(base['df'].y_type) | {'effect/phenotype'}
    assert set(d.G.node_types) == expect_ntypes, \
        'node types changed: %s' % sorted(expect_ntypes ^ set(d.G.node_types))

    set_all_seeds(seed)
    m = TxGNN(data=d, weight_bias_track=False, exp_name='%s_seed%d' % (arm_name, seed), device=DEVICE)
    m.model_initialize(**MODEL_CFG)

    # --- the signature is what we think it is ---------------------------------------------------
    sig_dim, sig_max = signature_probe(m.model)
    expect_dim = m.G['disease'].num_nodes + m.G['gene/protein'].num_nodes
    sig_sources = [r for r in SIG_ETYPES if r in rel_names]
    if sig_sources:
        assert sig_dim == expect_dim, 'signature dim %s != expected %s' % (sig_dim, expect_dim)
        assert sig_max == 1.0, \
            'signature should be populated for %s, got max %s' % (arm_name, sig_max)
    else:
        # dd_only: neither source relation survives, so every signature is the zero vector
        assert sig_max == 0.0, 'dd_only signature should be all-zero, got max %s' % sig_max

    enc_p, tot_p = encoder_params(m.model), get_n_params(m.model)
    print('[graph] train edges %d -> %d (%.2f%% kept) | edge types %d | node types %d'
          % (n_before, len(d.df_train), 100.0 * len(d.df_train) / n_before,
             len(etypes_after), n_node_types))
    print('[model] encoder params %d | total %d | signature dim %s max %s | sources %s'
          % (enc_p, tot_p, sig_dim, sig_max, sig_sources or 'NONE (degenerate)'))

    t0 = time.time()
    set_all_seeds(seed)
    m.pretrain(**PRETRAIN_CFG)
    pre_s = time.time() - t0

    t0 = time.time()
    set_all_seeds(seed)
    m.finetune(**FINETUNE_CFG)
    fin_s = time.time() - t0

    # ---- edge-level test metrics ----
    (auroc_rel, auprc_rel, micro_auroc, micro_auprc, macro_auroc, macro_auprc), test_loss = \
        evaluate_fb(m.best_model, m.g_test_pos, m.g_test_neg, m.G, m.dd_etypes, DEVICE, mode='test')

    el = pd.DataFrame(
        [('%s --%s--> %s' % et, auroc_rel.get(et, np.nan), auprc_rel.get(et, np.nan))
         for et in DD_ETYPES],
        columns=['relation', 'AUROC', 'AUPRC'])
    el.loc[len(el)] = ['MICRO (drug-disease)', micro_auroc, micro_auprc]
    el.loc[len(el)] = ['MACRO (drug-disease)',
                       float(np.nanmean([auroc_rel.get(et, np.nan) for et in DD_ETYPES])),
                       float(np.nanmean([auprc_rel.get(et, np.nan) for et in DD_ETYPES]))]
    el.loc[len(el)] = ['MACRO (all relations, NOT comparable across arms)', macro_auroc, macro_auprc]

    rdir.mkdir(parents=True, exist_ok=True)
    el.to_csv(el_csv, index=False)

    meta = {
        'run_key': '%s_seed%d' % (arm_name, seed),
        'arm': arm_name, 'seed': seed, 'split': SPLIT, 'profile': PROFILE,
        'experiment': 'keep_disease_only',
        'kept_relations_requested': arm['keep'],
        'kept_relation_names': sorted(rel_names),
        'n_edge_types_after': len(etypes_after),
        'edge_types_after': etypes_after,
        'n_node_types': n_node_types,
        'train_edges_before': n_before,
        'train_edges_after': len(d.df_train),
        'train_edges_kept_pct': 100.0 * len(d.df_train) / n_before,
        'dropped_edges_total': n_before - len(d.df_train),
        'signature_disease_etypes': SIG_ETYPES,
        'signature_sources_present': sig_sources,
        'signature_dim': sig_dim,
        'signature_max': sig_max,
        'encoder_params': enc_p,
        'n_params': tot_p,
        'proto': MODEL_CFG['proto'],
        'model_config': m.config,
        'pretrain': PRETRAIN_CFG, 'finetune': FINETUNE_CFG,
        'pretrain_seconds': pre_s, 'finetune_seconds': fin_s,
        'test_bce_loss': float(test_loss),
        'torch': torch.__version__,
        'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    # written last - is_done() needs both files, so a crash mid-write leaves the run pending
    with open(meta_json, 'w') as f:
        json.dump(meta, f, indent=2, default=str)

    if SAVE_MODELS:
        mdir = REPO_ROOT / 'saved_models' / 'ablation_keep_disease_only' / meta['run_key']
        mdir.mkdir(parents=True, exist_ok=True)
        m.save_model(str(mdir))

    ind = auprc_rel.get(('drug', 'indication', 'disease'), np.nan)
    con = auprc_rel.get(('drug', 'contraindication', 'disease'), np.nan)
    print('[done] %s_seed%d | %.1f min | indication AUPRC %.4f | contraindication AUPRC %.4f -> %s'
          % (arm_name, seed, (pre_s + fin_s) / 60, ind, con, rdir.relative_to(REPO_ROOT)))

    # ---- free the GPU before the next run ----
    del m, d, auroc_rel, auprc_rel
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return el

In [ ]:
# Seed-major: every arm at one seed before moving on, so a partial grid is always a set of complete
# paired comparisons rather than an unpaired reference. Also means one split load per seed.
if RUN_GRID:
    todo = [(a['arm'], s) for s in SEEDS for a in ARMS if not is_done(a['arm'], s)]
    if STOP_AFTER_N_RUNS:
        todo = todo[:STOP_AFTER_N_RUNS]

    print('%d run(s) pending of %d total.' % (len(todo), len(ARMS) * len(SEEDS)))
    grid_t0 = time.time()
    for i, (arm_name, seed) in enumerate(todo, 1):
        print('\n>>> %d/%d  %s_seed%d  (elapsed %.1f h)'
              % (i, len(todo), arm_name, seed, (time.time() - grid_t0) / 3600))
        run_one(arm_name, seed)
    print('\nGrid finished %d run(s) in %.1f h.' % (len(todo), (time.time() - grid_t0) / 3600))
else:
    print('RUN_GRID is False - no training. Set it in section 2.')

---
## 7 - Aggregate

Deltas are paired per seed against `full_kg`, the same statistic the other grids use. Reads whatever is
on disk, so it is safe against a partial grid.

In [ ]:
REF_ARM = 'full_kg'


def _load_all():
    recs = []
    for meta_p in sorted(ABL_ROOT.glob('*/run_meta.json')):
        el_p = meta_p.parent / 'edge_level_test_metrics.csv'
        if not el_p.exists():
            continue
        meta = json.loads(meta_p.read_text())
        el = pd.read_csv(el_p).set_index('relation')
        for rel, row in el.iterrows():
            recs.append({'arm': meta['arm'], 'seed': meta['seed'], 'relation': rel,
                         'AUROC': row['AUROC'], 'AUPRC': row['AUPRC'],
                         'edge_types': meta.get('n_edge_types_after'),
                         # imported reference runs record this as 'train_edges' instead
                         'train_edges': meta.get('train_edges_after', meta.get('train_edges')),
                         'encoder_params': meta.get('encoder_params'),
                         'signature_max': meta.get('signature_max'),
                         'imported': 'imported_from' in meta})
    return pd.DataFrame(recs)


def _ci95(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    return 1.96 * x.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan


ROW = {'indication': 'drug --indication--> disease',
       'contraindication': 'drug --contraindication--> disease',
       'micro': 'MICRO (drug-disease)'}

long = _load_all()
summary, wide, ref = None, None, None

if long.empty:
    print('No runs on disk yet - run the grid cell in section 6 first.')
else:
    wide = (long[long.relation.isin(ROW.values())]
            .pivot_table(index=['arm', 'seed'], columns='relation', values='AUPRC')
            .rename(columns={v: k for k, v in ROW.items()}))

    # imported reference runs predate this notebook's extra fields; take the first non-null per arm
    def _first(s):
        s = s.dropna()
        return s.iloc[0] if len(s) else np.nan

    info = long.groupby('arm').agg(edge_types=('edge_types', _first),
                                   train_edges=('train_edges', _first),
                                   encoder_params=('encoder_params', _first),
                                   signature_max=('signature_max', _first))

    if REF_ARM in wide.index.get_level_values('arm'):
        ref = wide.xs(REF_ARM, level='arm')
    else:
        print('[warn] no %s runs yet - deltas will be NaN.' % REF_ARM)

    out = []
    for arm, g in wide.groupby(level='arm'):
        g = g.droplevel('arm')
        rec = {'arm': arm, 'n_seeds': len(g),
               'edge_types': info.loc[arm, 'edge_types'],
               'train_edges': info.loc[arm, 'train_edges'],
               'encoder_params': info.loc[arm, 'encoder_params']}
        for key in ROW:
            rec['%s_AUPRC' % key] = g[key].mean()
            rec['%s_CI' % key] = _ci95(g[key])
            if arm == REF_ARM:
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = 0.0, np.nan
                rec['%s_n_paired' % key] = len(g)
            elif ref is not None:
                paired = (g[key] - ref[key]).dropna()      # inner-joins on seed
                rec['%s_delta' % key] = paired.mean() if len(paired) else np.nan
                rec['%s_delta_CI' % key] = _ci95(paired)
                rec['%s_n_paired' % key] = len(paired)
            else:
                rec['%s_delta' % key], rec['%s_delta_CI' % key] = np.nan, np.nan
                rec['%s_n_paired' % key] = 0
        out.append(rec)

    order = [a['arm'] for a in ARMS]
    summary = (pd.DataFrame(out)
                 .assign(_o=lambda d: d.arm.map({a: i for i, a in enumerate(order)}))
                 .sort_values('_o').drop(columns='_o').reset_index(drop=True))
    summary.to_csv(ABL_ROOT / 'keep_disease_only_summary.csv', index=False)

    cols = ['arm', 'n_seeds', 'edge_types', 'train_edges', 'encoder_params',
            'indication_AUPRC', 'indication_CI', 'indication_delta', 'indication_delta_CI',
            'contraindication_AUPRC', 'contraindication_CI', 'contraindication_delta',
            'contraindication_delta_CI', 'micro_AUPRC', 'micro_delta']
    print('%d arms, %d runs on disk -> %s\n'
          % (len(summary), len(wide), (ABL_ROOT / 'keep_disease_only_summary.csv').relative_to(REPO_ROOT)))
    display(summary[cols].style
                         .format('{:.4f}', subset=cols[5:], na_rep='-')
                         .format({'train_edges': '{:,.0f}', 'encoder_params': '{:,.0f}',
                                  'edge_types': '{:.0f}'})
                         .hide(axis='index'))

    short = summary[summary.n_seeds < len(SEEDS)]
    if len(short):
        print('\n[partial] below %d seeds, deltas provisional: %s'
              % (len(SEEDS), ', '.join('%s (n=%d)' % (r.arm, r.n_seeds) for r in short.itertuples())))

In [ ]:
# Per-seed paired deltas - the sign-consistency check a mean hides. 'wins' counts seeds on which the
# reduced graph actually hurt; 20/20 is a much stronger statement than a mean with a wide CI.
if summary is not None and ref is not None:
    top = [a for a in summary.arm if a != REF_ARM]
    for key in ['indication', 'contraindication']:
        tbl = pd.DataFrame({a: (wide.xs(a, level='arm')[key] - ref[key]) for a in top}).T
        seed_cols = list(tbl.columns)
        tbl.columns = ['seed %d' % s for s in seed_cols]
        tbl['mean'] = tbl.mean(axis=1)
        tbl['wins'] = (tbl[['seed %d' % s for s in seed_cols]] < 0).sum(axis=1)
        print('\npaired delta AUPRC vs full_kg - %s  (negative = the reduced graph hurt)' % key)
        display(tbl.style.format('{:.4f}', subset=[c for c in tbl.columns if c != 'wins'], na_rep='-'))

### 7.1 - Context: where does a 9-edge-type graph sit among the other ablations?

Pulls the headline paired deltas from the sibling grids and lists them next to this one, all measured
against a configuration-identical full-KG baseline on their own seeds. This is the table that says
whether "keep only the two disease relations" behaves like a small ablation or like gutting the model.

In [ ]:
import pathlib


def _grid_delta(root, arm, ref_root, ref_arm, metric_row):
    '''Paired delta for one arm of another grid, against that grid's own reference arm.'''
    def load(r, a):
        out = {}
        for mp in sorted(pathlib.Path(r).glob('%s_seed*/run_meta.json' % a)):
            ep = mp.parent / 'edge_level_test_metrics.csv'
            if not ep.exists():
                continue
            meta = json.loads(mp.read_text())
            if meta['arm'] != a:
                continue
            el = pd.read_csv(ep).set_index('relation')
            if metric_row in el.index:
                out[meta['seed']] = float(el.loc[metric_row, 'AUPRC'])
        return out
    A, B = load(root, arm), load(ref_root, ref_arm)
    common = sorted(set(A) & set(B))
    if not common:
        return np.nan, np.nan, 0
    d = np.array([A[s] - B[s] for s in common])
    ci = 1.96 * d.std(ddof=1) / np.sqrt(len(d)) if len(d) > 1 else np.nan
    return d.mean(), ci, len(d)


REFERENCES = [
    ('drop disease_protein (graph + signature)', REPLICATION_ROOT, 'drop_disease_protein_full',
     REPLICATION_ROOT, 'baseline'),
    ('drop disease_protein from signature only', SIGNATURE_ROOT, 'sig_drop_disease_protein',
     REPLICATION_ROOT, 'baseline'),
    ('no metric learning (proto_off)', SIGNATURE_ROOT, 'proto_off', REPLICATION_ROOT, 'baseline'),
    ('R-GCN -> shared-weight GNN', ENCODER_ROOT, 'light_gnn', ENCODER_ROOT, 'rgcn'),
    ('R-GCN -> MLP (no message passing)', ENCODER_ROOT, 'mlp', ENCODER_ROOT, 'rgcn'),
]

rows = []
if summary is not None and ref is not None:
    for arm in [a['arm'] for a in ARMS if a['arm'] != REF_ARM]:
        if arm not in summary.arm.values:
            continue
        r = summary.set_index('arm').loc[arm]
        rows.append({'change': 'THIS GRID: %s' % arm,
                     'indication delta': r['indication_delta'],
                     'contraindication delta': r['contraindication_delta'],
                     'micro delta': r['micro_delta'],
                     'n': int(r['indication_n_paired'])})
for label, root, arm, rroot, rarm in REFERENCES:
    di, ci_i, n = _grid_delta(root, arm, rroot, rarm, ROW['indication'])
    dc, _, _ = _grid_delta(root, arm, rroot, rarm, ROW['contraindication'])
    dm, _, _ = _grid_delta(root, arm, rroot, rarm, ROW['micro'])
    rows.append({'change': label, 'indication delta': di, 'contraindication delta': dc,
                 'micro delta': dm, 'n': n})

ctx = pd.DataFrame(rows).sort_values('indication delta')
if ctx.empty:
    print('Nothing to compare yet.')
else:
    ctx.to_csv(ABL_ROOT / 'context_vs_other_grids.csv', index=False)
    display(ctx.style.format('{:+.4f}', subset=['indication delta', 'contraindication delta',
                                                'micro delta'], na_rep='-').hide(axis='index'))
    print('\n-> %s' % (ABL_ROOT / 'context_vs_other_grids.csv').relative_to(REPO_ROOT))
    print('All deltas are paired per seed against a configuration-identical full-KG baseline.')

---
## 8 - Reading the result

### If `only_disease_relations` lands close to `full_kg`

Then the other 41 edge types - 96% of the training edges, `drug_drug`, `protein_protein`,
`anatomy_protein_present` and the rest - contribute nothing that message passing can use, and TxGNN's
knowledge graph reduces to `disease_disease` + `disease_protein` + the task labels. That is a strong,
checkable, deflationary claim, and it goes well beyond the leave-one-out result because leave-one-out
cannot see redundancy: 41 relations could each be individually worthless yet collectively essential.
This arm closes that gap.

### If it loses substantially

Be careful before concluding the dropped relations carry information, because **two things changed at
once**:

1. **Encoder capacity.** `HeteroRGCNLayer` allocates one `nn.Linear(100, 100)` per relation, so the
   encoder shrinks from 1,010,000 parameters to 181,800 - 18% of the baseline. The encoder-swap grid
   showed this metric is very sensitive to encoder capacity (collapsing 50 relation matrices into one
   shared matrix cost -0.2266 indication at a *fixed* graph). Some of any loss here is that, not missing
   KG information. **The control:** rerun `only_disease_relations` with `n_hid`/`n_out` widened until
   `encoder_params` matches 1,010,000 (with 9 relations that needs `n_hid = n_out = 288`, which gives
   1,010,880), and report both.
2. **Relation identity vs relation count.** A 9-edge-type graph might do poorly simply for being small.
   **The control:** an arm keeping the task labels plus three arbitrary non-disease relations of
   comparable size - e.g. `pathway_protein`, `cellcomp_protein`, `molfunc_protein` (85k/83k/68k train
   edges each way, against `disease_protein`'s 67k) - giving the same 9 edge types and the same encoder
   size. If `only_disease_relations` beats that, the identity of the two relations is what matters; if
   they tie, it is the count.

Both are a handful of lines on top of this notebook and about 2 GPU-hours each at these edge counts.

### Reading `dd_only`

It is the floor, but a doubly degenerate one: no KG context *and* an all-zero signature, since neither
signature source relation survives (asserted in `run_one`, logged as `signature_max = 0.0`). So it
measures "DistMult on the task edges with a 6-relation encoder and a dead prototype gate", not "TxGNN
without a KG". For a clean no-metric-learning reference use `proto_off` from
`ablation_signature_only.ipynb` (-0.1737 indication, 10/10 seeds); for a clean no-KG-context reference,
an arm with `keep=DD_RELATIONS` *and* `proto=False` would be more interpretable than this one.

### Carry-over caveats from the other grids

- Edge-level AUPRC samples one negative per positive, so **chance is ~0.500**, not 0. Read every absolute
  number against that floor.
- This reproduction's full model sits at indication 0.835 / contraindication 0.734 against the paper's
  0.89 / 0.80, so report relative deltas, not absolute agreement.
- `MACRO (all relations)` is not comparable across arms here (50 vs 9 vs 6 edge types in the mean).